In [ ]:
# %% [markdown]
# # 01 — Comparação de modelos
#
# **TCC — Adoecimento de professores da educação básica (RAIS)**
#
# Este programa compara Dummy, regressão logística, Random Forest, XGBoost,
# LightGBM e CatBoost utilizando:
#
# - por padrão, a base completa de 2020–2023 (a amostra de 800 mil continua
#   disponível como modo de teste e análise de sensibilidade);
# - 2020–2022 para ajuste inicial dos modelos de boosting;
# - 2023 para early stopping e escolha do número de árvores;
# - novo treinamento em 2020–2023 com o número de árvores congelado;
# - validação completa em 2024;
# - nenhum acesso aos dados de 2025.
#
# A métrica principal para ordenar os modelos é **Average Precision (AP)**.

# %% [markdown]
# ## 1. Instalação das bibliotecas
#
# Esta etapa instala ou atualiza as bibliotecas no Google Colab.

# %%
import subprocess
import sys

PACOTES = [
    "xgboost",
    "lightgbm",
    "catboost",
    "pyarrow",
    "joblib",
]

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-U", *PACOTES]
)

# %% [markdown]
# ## 2. Importações e Google Drive

# %%
from google.colab import drive

import gc
import glob
import json
import os
import re
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from catboost import CatBoostClassifier
import lightgbm as lgb
from lightgbm import LGBMClassifier
import xgboost as xgb
from xgboost import XGBClassifier

from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore", category=FutureWarning)

drive.mount("/content/drive", force_remount=False)

print("Versões:")
print("xgboost:", xgb.__version__)
print("lightgbm:", lgb.__version__)

# %% [markdown]
# ## 3. Pastas e configurações
#
# Por padrão, o código procura os arquivos anuais de 2020–2024 dentro da base
# final V2. No modo de amostra, procura `02b_amostra_aninhada_800k.parquet`.

# %%
SEED = 42
TARGET = "Y_doenca"

ANOS_DESENVOLVIMENTO = [2020, 2021, 2022]
ANO_EARLY_STOPPING = 2023
ANO_VALIDACAO = 2024

BATCH_SIZE_VALIDACAO = 200_000
BATCH_SIZE_LEITURA_TREINO = 500_000
RECALL_ALVO = 0.70

# "BASE_COMPLETA" = todos os registros de 2020–2023.
# "AMOSTRA_800K" = execução mais rápida e análise de sensibilidade.
MODO_DADOS = "BASE_COMPLETA"

# Cada árvore do RF usa uma subamostra. Na base completa, 25% ainda representa
# milhões de vínculos por árvore e reduz substancialmente tempo e memória.
RF_MAX_SAMPLES = 0.25 if MODO_DADOS == "BASE_COMPLETA" else 0.70

# Para Colab com pouca memória, execute um modelo por vez, preservando os
# resultados já concluídos. Exemplo: ["LGBM"].
MODELOS_EXECUTAR = ["DUMMY", "LOGIT", "RF", "XGB", "LGBM", "CAT"]

# False permite retomar a execução: modelos já concluídos serão reutilizados.
# Altere para True somente se quiser refazer os resultados existentes.
SOBRESCREVER_RESULTADOS = False

PASTA_TCC = "/content/drive/MyDrive/TCC_2"
PASTA_BASE = os.path.join(
    PASTA_TCC,
    "dados",
    "RAIS_BASE_MODELO_FINAL_V2",
)
PASTA_RESULTADOS = os.path.join(
    PASTA_TCC,
    "resultados",
    f"ML_COMPARACAO_MODELOS_V2_{MODO_DADOS}",
)

PASTA_MODELOS = os.path.join(PASTA_RESULTADOS, "modelos")
PASTA_PREDICOES = os.path.join(PASTA_RESULTADOS, "predicoes_2024")

for pasta in [PASTA_RESULTADOS, PASTA_MODELOS, PASTA_PREDICOES]:
    os.makedirs(pasta, exist_ok=True)

if MODO_DADOS not in {"BASE_COMPLETA", "AMOSTRA_800K"}:
    raise ValueError("MODO_DADOS deve ser BASE_COMPLETA ou AMOSTRA_800K.")

ARQUIVO_AMOSTRA = None
if MODO_DADOS == "AMOSTRA_800K":
    padrao_amostra = os.path.join(
        PASTA_TCC, "**", "02b_amostra_aninhada_800k.parquet"
    )
    amostras_encontradas = glob.glob(padrao_amostra, recursive=True)
    if not amostras_encontradas:
        raise FileNotFoundError(
            "Não encontrei 02b_amostra_aninhada_800k.parquet dentro de TCC_2."
        )
    ARQUIVO_AMOSTRA = max(amostras_encontradas, key=os.path.getmtime)
    print("Amostra selecionada:")
    print(ARQUIVO_AMOSTRA)
else:
    print("Modo selecionado: base completa de 2020–2023.")
print("\nPasta dos novos resultados:")
print(PASTA_RESULTADOS)

# %% [markdown]
# ## 4. Preditores da especificação final
#
# A escolaridade permanece excluída. Os dois vínculos públicos permanecem separados.
# `Ano` é utilizado somente para a separação temporal e não entra como preditor.

# %%
PREDITORES_NUMERICOS = [
    "Idade",
    "Qtd_horas_contratuais",
    "Tempo_emprego_meses",
]

PREDITORES_CATEGORICOS = [
    "UF",
    "Familia_CBO",
    "Sexo_codigo",
    "Tipo_vinculo_macro",
    "Natureza_macro",
    "Tamanho_estabelecimento_codigo",
    "Indicador_deficiencia_codigo",
]

PREDITORES = PREDITORES_NUMERICOS + PREDITORES_CATEGORICOS
COLUNAS_AMOSTRA = ["Ano"] + PREDITORES + [TARGET]
COLUNAS_VALIDACAO = PREDITORES + [TARGET]

def limpar_tipo_vinculo(serie):
    """Padroniza espaços, mas NÃO reúne as categorias de vínculo público."""
    return (
        serie.astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )


def preparar_preditores_base(df):
    """Harmonização comum aos modelos, sem aprender com a validação."""
    X = df[PREDITORES].copy()

    for coluna in PREDITORES_NUMERICOS:
        X[coluna] = pd.to_numeric(X[coluna], errors="coerce")

    for coluna in PREDITORES_CATEGORICOS:
        serie = X[coluna]
        if isinstance(serie.dtype, pd.CategoricalDtype):
            # O treino completo já foi harmonizado com categorias globais.
            X[coluna] = serie
            continue
        if coluna == "Tipo_vinculo_macro":
            serie = limpar_tipo_vinculo(serie)
        X[coluna] = (
            serie.astype("string")
            .fillna("__MISSING__")
            .astype("category")
        )

    return X


def validar_colunas(colunas_disponiveis, colunas_necessarias, origem):
    faltantes = sorted(set(colunas_necessarias) - set(colunas_disponiveis))
    if faltantes:
        raise ValueError(f"Colunas ausentes em {origem}: {faltantes}")

# %% [markdown]
# ## 5. Carregar e auditar os dados de treinamento
#
# No modo principal, todos os arquivos de 2020–2023 são lidos em lotes. As
# colunas são reduzidas aos tipos necessários para conter o uso de memória.
# A alternativa de 800 mil permanece disponível para testes rápidos.

# %%
def reduzir_tipos_treino(df, ano_padrao=None, categorias_globais=None):
    """Padroniza tipos sem utilizar informação de 2024 ou 2025."""
    saida = df[COLUNAS_AMOSTRA].copy()
    if ano_padrao is not None:
        saida["Ano"] = ano_padrao
    saida["Ano"] = pd.to_numeric(saida["Ano"], errors="raise").astype("int16")
    saida[TARGET] = pd.to_numeric(saida[TARGET], errors="raise").astype("int8")
    for coluna in PREDITORES_NUMERICOS:
        saida[coluna] = pd.to_numeric(
            saida[coluna], errors="coerce"
        ).astype("float32")
    for coluna in PREDITORES_CATEGORICOS:
        serie = saida[coluna]
        if coluna == "Tipo_vinculo_macro":
            serie = limpar_tipo_vinculo(serie)
        serie = serie.astype("string").fillna("__MISSING__")
        if categorias_globais is None:
            saida[coluna] = serie
        else:
            saida[coluna] = pd.Categorical(
                serie,
                categories=categorias_globais[coluna],
            )
    return saida


def localizar_arquivos_anos(anos):
    encontrados = []
    por_ano = {}
    for ano in anos:
        arquivos = sorted(
            glob.glob(
                os.path.join(
                    PASTA_BASE,
                    "**",
                    f"RAIS_MODELO_FINAL_V2_{ano}_*.parquet",
                ),
                recursive=True,
            )
        )
        if not arquivos:
            raise FileNotFoundError(f"Nenhum arquivo de {ano} foi encontrado.")
        por_ano[ano] = arquivos
        encontrados.extend((ano, arquivo) for arquivo in arquivos)
    return encontrados, por_ano


def descobrir_categorias_treino(arquivos_ano):
    """Primeira passagem leve para manter as categóricas compactadas."""
    conjuntos = {c: {"__MISSING__"} for c in PREDITORES_CATEGORICOS}
    for ano, arquivo in arquivos_ano:
        parquet = pq.ParquetFile(arquivo)
        validar_colunas(parquet.schema.names, PREDITORES_CATEGORICOS, arquivo)
        for batch in parquet.iter_batches(
            batch_size=BATCH_SIZE_LEITURA_TREINO,
            columns=PREDITORES_CATEGORICOS,
        ):
            parte = batch.to_pandas()
            for coluna in PREDITORES_CATEGORICOS:
                serie = parte[coluna]
                if coluna == "Tipo_vinculo_macro":
                    serie = limpar_tipo_vinculo(serie)
                valores = serie.astype("string").fillna("__MISSING__").unique()
                conjuntos[coluna].update(map(str, valores.tolist()))
            del parte
        print(f"Categorias auditadas | {ano} | {os.path.basename(arquivo)}")
    return {coluna: sorted(valores) for coluna, valores in conjuntos.items()}


def carregar_base_completa():
    arquivos_ano, mapa = localizar_arquivos_anos(
        ANOS_DESENVOLVIMENTO + [ANO_EARLY_STOPPING]
    )
    categorias_globais = descobrir_categorias_treino(arquivos_ano)
    partes = []
    total = 0
    for ano, arquivo in arquivos_ano:
        schema = pq.ParquetFile(arquivo).schema.names
        # Alguns arquivos anuais não guardam Ano porque ele já está no nome.
        colunas_arquivo = [c for c in COLUNAS_AMOSTRA if c in schema]
        validar_colunas(
            schema,
            PREDITORES + [TARGET],
            arquivo,
        )
        parquet = pq.ParquetFile(arquivo)
        for batch in parquet.iter_batches(
            batch_size=BATCH_SIZE_LEITURA_TREINO,
            columns=colunas_arquivo,
        ):
            parte = batch.to_pandas()
            if "Ano" not in parte.columns:
                parte["Ano"] = ano
            parte = reduzir_tipos_treino(
                parte,
                ano_padrao=ano,
                categorias_globais=categorias_globais,
            )
            partes.append(parte)
            total += len(parte)
            print(f"Treino | {ano} | {os.path.basename(arquivo)} | {total:,}")

    df = pd.concat(partes, ignore_index=True)
    del partes
    gc.collect()

    return df, mapa


if MODO_DADOS == "BASE_COMPLETA":
    df_treino, ARQUIVOS_TREINO_POR_ANO = carregar_base_completa()
else:
    schema_amostra = pq.ParquetFile(ARQUIVO_AMOSTRA).schema.names
    validar_colunas(schema_amostra, COLUNAS_AMOSTRA, ARQUIVO_AMOSTRA)
    df_treino = pd.read_parquet(ARQUIVO_AMOSTRA, columns=COLUNAS_AMOSTRA)
    df_treino = reduzir_tipos_treino(df_treino)
    if len(df_treino) != 800_000:
        raise ValueError(
            f"A amostra deveria ter 800.000 registros, mas possui {len(df_treino):,}."
        )
    ARQUIVOS_TREINO_POR_ANO = None

anos_treino = sorted(df_treino["Ano"].unique().tolist())
if anos_treino != [2020, 2021, 2022, 2023]:
    raise ValueError(f"Anos inesperados no treino: {anos_treino}")
if not set(df_treino[TARGET].unique()).issubset({0, 1}):
    raise ValueError("Y_doenca deve conter somente 0 e 1.")

auditoria_amostra = (
    df_treino.groupby(["Ano", TARGET], observed=True)
    .size()
    .rename("N")
    .reset_index()
)
auditoria_amostra["Proporcao_no_ano"] = (
    auditoria_amostra["N"]
    / auditoria_amostra.groupby("Ano")["N"].transform("sum")
)
auditoria_amostra.to_csv(
    os.path.join(PASTA_RESULTADOS, "00_auditoria_treino.csv"),
    index=False,
    encoding="utf-8-sig",
)

display(auditoria_amostra)

X_amostra = preparar_preditores_base(df_treino)
y_amostra = df_treino[TARGET].to_numpy(dtype=np.int8)
anos = df_treino["Ano"].to_numpy(dtype=np.int16)

mascara_dev = np.isin(anos, ANOS_DESENVOLVIMENTO)
mascara_2023 = anos == ANO_EARLY_STOPPING

X_dev = X_amostra.loc[mascara_dev].reset_index(drop=True)
y_dev = y_amostra[mascara_dev]

X_2023 = X_amostra.loc[mascara_2023].reset_index(drop=True)
y_2023 = y_amostra[mascara_2023]

X_treino_completo = X_amostra.reset_index(drop=True)
y_treino_completo = y_amostra

print(f"Desenvolvimento 2020–2022: {len(X_dev):,}")
print(f"Early stopping 2023:       {len(X_2023):,}")
print(f"Treino final 2020–2023:   {len(X_treino_completo):,}")
print(f"Prevalência no treino:    {y_treino_completo.mean():.4%}")

categorias_vinculo = sorted(
    X_treino_completo["Tipo_vinculo_macro"].astype(str).unique().tolist()
)
print("\nCategorias de Tipo_vinculo_macro:")
for categoria in categorias_vinculo:
    print("-", categoria)

publicos = [
    c for c in categorias_vinculo
    if c.casefold().startswith(("público", "publico"))
]
if len(publicos) < 2:
    raise ValueError(
        "Foram encontradas menos de duas categorias públicas. "
        "Revise a harmonização antes de continuar."
    )

memoria_mb = df_treino.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"Memória do quadro de treino: {memoria_mb:,.1f} MB")

del df_treino, X_amostra
gc.collect()

# %% [markdown]
# ## 6. Localizar exclusivamente os arquivos de 2024
#
# A expressão regular identifica o ano pelo nome do arquivo. A lista de 2025
# não é criada e nenhum arquivo desse ano é aberto.

# %%
arquivos_validacao = sorted(
    glob.glob(
        os.path.join(
            PASTA_BASE,
            "**",
            f"RAIS_MODELO_FINAL_V2_{ANO_VALIDACAO}_*.parquet",
        ),
        recursive=True,
    )
)

if not arquivos_validacao:
    raise FileNotFoundError(f"Nenhum arquivo de {ANO_VALIDACAO} foi encontrado.")

for caminho in arquivos_validacao:
    validar_colunas(
        pq.ParquetFile(caminho).schema.names,
        COLUNAS_VALIDACAO,
        caminho,
    )

print(f"Arquivos utilizados na validação de 2024: {len(arquivos_validacao)}")
for caminho in arquivos_validacao:
    print("-", os.path.basename(caminho))
print("\n2025 permanece lacrado e não será lido neste notebook.")

# %% [markdown]
# ## 7. Funções compartilhadas

# %%
def razao_desbalanceamento(y):
    positivos = int(np.sum(y == 1))
    negativos = int(np.sum(y == 0))
    if positivos == 0:
        raise ValueError("O treino não possui casos positivos.")
    return negativos / positivos


def criar_one_hot():
    parametros = dict(handle_unknown="ignore", dtype=np.float32)
    try:
        return OneHotEncoder(sparse_output=True, **parametros)
    except TypeError:
        return OneHotEncoder(sparse=True, **parametros)


def criar_preprocessador_one_hot():
    pipeline_numerico = Pipeline(
        [("imputacao", SimpleImputer(strategy="median"))]
    )
    pipeline_categorico = Pipeline(
        [("one_hot", criar_one_hot())]
    )
    return ColumnTransformer(
        transformers=[
            ("num", pipeline_numerico, PREDITORES_NUMERICOS),
            ("cat", pipeline_categorico, PREDITORES_CATEGORICOS),
        ],
        remainder="drop",
        sparse_threshold=1.0,
    )


def salvar_dicionario_dummies(preprocessador, nome_modelo):
    """Registra nomes como cat__UF_SP e cat__Familia_CBO_2312."""
    nomes = preprocessador.get_feature_names_out()
    tabela = pd.DataFrame(
        {
            "Posicao": np.arange(len(nomes), dtype=int),
            "Feature_transformada": nomes,
        }
    )
    tabela.to_csv(
        os.path.join(
            PASTA_RESULTADOS,
            f"dicionario_features_one_hot_{nome_modelo}.csv",
        ),
        index=False,
        encoding="utf-8-sig",
    )


def aprender_metadados_nativos(X):
    medianas = {
        coluna: float(pd.to_numeric(X[coluna], errors="coerce").median())
        for coluna in PREDITORES_NUMERICOS
    }
    categorias = {
        coluna: sorted(X[coluna].astype(str).unique().tolist())
        for coluna in PREDITORES_CATEGORICOS
    }
    return {"medianas": medianas, "categorias": categorias}


def preparar_lightgbm(X, metadados):
    saida = X[PREDITORES].copy()
    for coluna in PREDITORES_NUMERICOS:
        saida[coluna] = (
            pd.to_numeric(saida[coluna], errors="coerce")
            .fillna(metadados["medianas"][coluna])
            .astype("float32")
        )
    for coluna in PREDITORES_CATEGORICOS:
        valores = saida[coluna].astype("string").fillna("__MISSING__")
        saida[coluna] = pd.Categorical(
            valores,
            categories=metadados["categorias"][coluna],
        )
    return saida


def preparar_catboost(X, metadados):
    saida = X[PREDITORES].copy()
    for coluna in PREDITORES_NUMERICOS:
        saida[coluna] = (
            pd.to_numeric(saida[coluna], errors="coerce")
            .fillna(metadados["medianas"][coluna])
            .astype("float32")
        )
    for coluna in PREDITORES_CATEGORICOS:
        saida[coluna] = (
            saida[coluna].astype("string").fillna("__MISSING__").astype(str)
        )
    return saida


def prever_bundle(bundle, X):
    tipo = bundle["tipo"]
    if tipo in {"LOGIT", "RF", "XGB"}:
        matriz = bundle["preprocessador"].transform(X)
        probabilidades = bundle["modelo"].predict_proba(matriz)[:, 1]
        del matriz
    elif tipo == "DUMMY":
        entrada = np.zeros((len(X), 1), dtype=np.float32)
        probabilidades = bundle["modelo"].predict_proba(entrada)[:, 1]
        del entrada
    elif tipo == "LGBM":
        dados = preparar_lightgbm(X, bundle["metadados"])
        probabilidades = bundle["modelo"].predict_proba(dados)[:, 1]
        del dados
    elif tipo == "CAT":
        dados = preparar_catboost(X, bundle["metadados"])
        probabilidades = bundle["modelo"].predict_proba(dados)[:, 1]
        del dados
    else:
        raise ValueError(f"Tipo de modelo desconhecido: {tipo}")
    return np.asarray(probabilidades, dtype=np.float32)


def calcular_limiares(y, prob):
    precisao_pr, recall_pr, thresholds_pr = precision_recall_curve(y, prob)
    denominador = precisao_pr[:-1] + recall_pr[:-1]
    f1_pr = np.divide(
        2 * precisao_pr[:-1] * recall_pr[:-1],
        denominador,
        out=np.zeros_like(denominador),
        where=denominador > 0,
    )
    indice_f1 = int(np.nanargmax(f1_pr))
    threshold_f1 = float(thresholds_pr[indice_f1])

    fpr, tpr, thresholds_roc = roc_curve(y, prob)
    indice_youden = int(np.nanargmax(tpr - fpr))
    threshold_youden = float(thresholds_roc[indice_youden])

    elegiveis = np.flatnonzero(recall_pr[:-1] >= RECALL_ALVO)
    if len(elegiveis):
        indice_recall = elegiveis[np.argmax(thresholds_pr[elegiveis])]
        threshold_recall = float(thresholds_pr[indice_recall])
    else:
        threshold_recall = 0.0

    return {
        "Threshold_0.50": 0.50,
        "Threshold_F1_validacao_2024": threshold_f1,
        "Threshold_Youden_validacao_2024": threshold_youden,
        "Threshold_recall_minimo_70pct": threshold_recall,
    }


def calcular_metricas(y, prob, threshold, modelo, configuracao):
    pred = (prob >= threshold).astype(np.int8)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    especificidade = tn / (tn + fp) if (tn + fp) else np.nan
    return {
        "Modelo": modelo,
        "Configuracao": configuracao,
        "Threshold": float(threshold),
        "N_validacao": int(len(y)),
        "Prevalencia_2024": float(np.mean(y)),
        "Accuracy": accuracy_score(y, pred),
        "Balanced_accuracy": balanced_accuracy_score(y, pred),
        "Precision": precision_score(y, pred, zero_division=0),
        "Recall_sensibilidade": recall_score(y, pred, zero_division=0),
        "Especificidade": especificidade,
        "F1": f1_score(y, pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y, prob),
        "Average_Precision_PR": average_precision_score(y, prob),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
    }


ARQUIVO_METRICAS = os.path.join(
    PASTA_RESULTADOS, "01_metricas_modelos_2024.csv"
)


def atualizar_metricas(linhas_novas, modelo):
    novas = pd.DataFrame(linhas_novas)
    if os.path.exists(ARQUIVO_METRICAS):
        anteriores = pd.read_csv(ARQUIVO_METRICAS)
        if "Modelo" in anteriores.columns:
            anteriores = anteriores.loc[anteriores["Modelo"] != modelo]
        tabela = pd.concat([anteriores, novas], ignore_index=True)
    else:
        tabela = novas
    tabela.to_csv(ARQUIVO_METRICAS, index=False, encoding="utf-8-sig")
    return tabela


def avaliar_modelo_2024(nome_modelo, bundle, tempo_treino_segundos, detalhes):
    caminho_prob = os.path.join(
        PASTA_PREDICOES, f"probabilidades_2024_{nome_modelo}.npy"
    )
    caminho_y = os.path.join(PASTA_PREDICOES, "y_real_2024.npy")
    caminho_info = os.path.join(
        PASTA_RESULTADOS, f"info_{nome_modelo}.json"
    )

    y_partes = []
    prob_partes = []
    total = 0
    inicio = time.time()

    for numero_arquivo, arquivo in enumerate(arquivos_validacao, start=1):
        parquet = pq.ParquetFile(arquivo)
        for numero_batch, batch in enumerate(
            parquet.iter_batches(
                batch_size=BATCH_SIZE_VALIDACAO,
                columns=COLUNAS_VALIDACAO,
            ),
            start=1,
        ):
            df_batch = batch.to_pandas()
            y_batch = pd.to_numeric(
                df_batch[TARGET], errors="raise"
            ).to_numpy(dtype=np.int8)
            X_batch = preparar_preditores_base(df_batch)
            prob_batch = prever_bundle(bundle, X_batch)

            y_partes.append(y_batch)
            prob_partes.append(prob_batch)
            total += len(y_batch)

            del df_batch, X_batch, y_batch, prob_batch
            gc.collect()

            print(
                f"{nome_modelo} | arquivo {numero_arquivo}/{len(arquivos_validacao)} "
                f"| batch {numero_batch} | {total:,} registros"
            )

    y_val = np.concatenate(y_partes).astype(np.int8, copy=False)
    prob_val = np.concatenate(prob_partes).astype(np.float32, copy=False)
    tempo_validacao = time.time() - inicio

    if os.path.exists(caminho_y):
        y_anterior = np.load(caminho_y, mmap_mode="r")
        if len(y_anterior) != len(y_val) or not np.array_equal(y_anterior, y_val):
            raise RuntimeError("A ordem de Y em 2024 mudou entre os modelos.")
        del y_anterior
    else:
        np.save(caminho_y, y_val)

    np.save(caminho_prob, prob_val)

    limiares = calcular_limiares(y_val, prob_val)
    linhas = [
        calcular_metricas(
            y_val,
            prob_val,
            threshold,
            nome_modelo,
            nome_configuracao,
        )
        for nome_configuracao, threshold in limiares.items()
    ]
    tabela_metricas = atualizar_metricas(linhas, nome_modelo)

    informacoes = {
        "modelo": nome_modelo,
        "modo_dados": MODO_DADOS,
        "seed": SEED,
        "anos_desenvolvimento": ANOS_DESENVOLVIMENTO,
        "ano_early_stopping": ANO_EARLY_STOPPING,
        "ano_validacao": ANO_VALIDACAO,
        "ano_2025_lido": False,
        "n_treino_final": int(len(y_treino_completo)),
        "prevalencia_treino": float(np.mean(y_treino_completo)),
        "n_validacao_2024": int(len(y_val)),
        "prevalencia_2024": float(np.mean(y_val)),
        "tempo_treino_segundos": float(tempo_treino_segundos),
        "tempo_validacao_segundos": float(tempo_validacao),
        "limiares": limiares,
        "detalhes": detalhes,
    }
    with open(caminho_info, "w", encoding="utf-8") as arquivo_json:
        json.dump(informacoes, arquivo_json, ensure_ascii=False, indent=2)

    del y_partes, prob_partes, y_val, prob_val
    gc.collect()

    display(
        tabela_metricas.loc[tabela_metricas["Modelo"] == nome_modelo]
        .sort_values("Configuracao")
        .reset_index(drop=True)
    )


def resultado_modelo_existe(nome_modelo):
    arquivos_ok = (
        os.path.exists(os.path.join(PASTA_MODELOS, f"{nome_modelo}.joblib"))
        and os.path.exists(
            os.path.join(PASTA_PREDICOES, f"probabilidades_2024_{nome_modelo}.npy")
        )
        and os.path.exists(ARQUIVO_METRICAS)
    )
    if not arquivos_ok:
        return False
    metricas_existentes = pd.read_csv(ARQUIVO_METRICAS, usecols=["Modelo"])
    return nome_modelo in set(metricas_existentes["Modelo"].astype(str))


def deve_executar(nome_modelo):
    if nome_modelo not in MODELOS_EXECUTAR:
        print(f"{nome_modelo}: não selecionado.")
        return False
    if resultado_modelo_existe(nome_modelo) and not SOBRESCREVER_RESULTADOS:
        print(f"{nome_modelo}: resultado existente; etapa ignorada.")
        return False
    return True

# %% [markdown]
# ## 8. Baseline Dummy
#
# O baseline prevê a prevalência observada no treino. Um modelo útil deve
# superar sua Average Precision, que tende a ficar próxima da prevalência.

# %%
if deve_executar("DUMMY"):
    inicio = time.time()
    entrada_dummy = np.zeros((len(y_treino_completo), 1), dtype=np.uint8)
    modelo_dummy = DummyClassifier(strategy="prior", random_state=SEED)
    modelo_dummy.fit(entrada_dummy, y_treino_completo)
    tempo_dummy = time.time() - inicio
    del entrada_dummy

    bundle_dummy = {
        "tipo": "DUMMY",
        "modelo": modelo_dummy,
        "preditores": [],
    }
    joblib.dump(
        bundle_dummy,
        os.path.join(PASTA_MODELOS, "DUMMY.joblib"),
        compress=3,
    )
    avaliar_modelo_2024(
        "DUMMY",
        bundle_dummy,
        tempo_dummy,
        {"strategy": "prior", "finalidade": "baseline de prevalência"},
    )
    del bundle_dummy, modelo_dummy
    gc.collect()

# %% [markdown]
# ## 9. Regressão logística regularizada
#
# A regressão logística é estimada por gradiente estocástico, solução mais
# escalável para a matriz one-hot da base completa. Continua sendo um modelo
# linear logístico com regularização L2 e fornece um baseline interpretável.

# %%
if deve_executar("LOGIT"):
    inicio = time.time()
    preprocessador_logit = criar_preprocessador_one_hot()
    X_logit = preprocessador_logit.fit_transform(X_treino_completo)

    modelo_logit = SGDClassifier(
        loss="log_loss",
        penalty="l2",
        alpha=1e-5,
        max_iter=100,
        tol=1e-3,
        class_weight="balanced",
        average=True,
        random_state=SEED,
        n_jobs=-1,
    )
    modelo_logit.fit(X_logit, y_treino_completo)
    salvar_dicionario_dummies(preprocessador_logit, "LOGIT")
    tempo_logit = time.time() - inicio

    bundle_logit = {
        "tipo": "LOGIT",
        "preprocessador": preprocessador_logit,
        "modelo": modelo_logit,
        "preditores": PREDITORES,
    }
    joblib.dump(
        bundle_logit,
        os.path.join(PASTA_MODELOS, "LOGIT.joblib"),
        compress=3,
    )
    del X_logit
    gc.collect()

    avaliar_modelo_2024(
        "LOGIT",
        bundle_logit,
        tempo_logit,
        {
            "estimador": "SGDClassifier com perda logística",
            "penalty": "l2",
            "alpha": 1e-5,
            "class_weight": "balanced",
            "average": True,
            "codificacao": "one-hot",
        },
    )
    del bundle_logit, modelo_logit, preprocessador_logit
    gc.collect()

# %% [markdown]
# ## 10. Random Forest
#
# O RF funciona como referência. Ele é treinado diretamente em 2020–2023,
# pois não utiliza early stopping.

# %%
if deve_executar("RF"):
    inicio = time.time()

    preprocessador_rf = criar_preprocessador_one_hot()
    X_rf = preprocessador_rf.fit_transform(X_treino_completo)

    modelo_rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=24,
        min_samples_split=40,
        min_samples_leaf=20,
        max_features="sqrt",
        max_samples=RF_MAX_SAMPLES,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=SEED,
        verbose=1,
    )
    modelo_rf.fit(X_rf, y_treino_completo)
    salvar_dicionario_dummies(preprocessador_rf, "RF")
    tempo_rf = time.time() - inicio

    bundle_rf = {
        "tipo": "RF",
        "preprocessador": preprocessador_rf,
        "modelo": modelo_rf,
        "preditores": PREDITORES,
    }
    joblib.dump(
        bundle_rf,
        os.path.join(PASTA_MODELOS, "RF.joblib"),
        compress=3,
    )

    del X_rf
    gc.collect()

    avaliar_modelo_2024(
        "RF",
        bundle_rf,
        tempo_rf,
        {
            "n_estimators": 200,
            "max_depth": 24,
            "min_samples_split": 40,
            "min_samples_leaf": 20,
            "max_samples": RF_MAX_SAMPLES,
            "class_weight": "balanced_subsample",
            "codificacao": "one-hot",
        },
    )

    del bundle_rf, modelo_rf, preprocessador_rf
    gc.collect()

# %% [markdown]
# ## 11. XGBoost
#
# Primeiro encontra o número de árvores com 2023. Depois cria um novo
# preprocessador e reestima o modelo em todo o conjunto de treino selecionado.

# %%
if deve_executar("XGB"):
    peso_xgb_dev = razao_desbalanceamento(y_dev)

    preprocessador_xgb_dev = criar_preprocessador_one_hot()
    X_xgb_dev = preprocessador_xgb_dev.fit_transform(X_dev)
    X_xgb_2023 = preprocessador_xgb_dev.transform(X_2023)

    modelo_xgb_dev = XGBClassifier(
        objective="binary:logistic",
        eval_metric="aucpr",
        n_estimators=1500,
        learning_rate=0.05,
        max_depth=8,
        min_child_weight=20,
        subsample=0.80,
        colsample_bytree=0.80,
        reg_lambda=1.0,
        max_bin=256,
        tree_method="hist",
        scale_pos_weight=peso_xgb_dev,
        early_stopping_rounds=60,
        n_jobs=-1,
        random_state=SEED,
    )
    modelo_xgb_dev.fit(
        X_xgb_dev,
        y_dev,
        eval_set=[(X_xgb_2023, y_2023)],
        verbose=100,
    )

    melhor_iteracao_xgb = getattr(modelo_xgb_dev, "best_iteration", None)
    n_arvores_xgb = (
        int(melhor_iteracao_xgb) + 1
        if melhor_iteracao_xgb is not None
        else 1500
    )
    print("Número de árvores escolhido para XGB:", n_arvores_xgb)

    del (
        preprocessador_xgb_dev,
        X_xgb_dev,
        X_xgb_2023,
        modelo_xgb_dev,
    )
    gc.collect()

    inicio = time.time()
    preprocessador_xgb = criar_preprocessador_one_hot()
    X_xgb = preprocessador_xgb.fit_transform(X_treino_completo)

    modelo_xgb = XGBClassifier(
        objective="binary:logistic",
        eval_metric="aucpr",
        n_estimators=n_arvores_xgb,
        learning_rate=0.05,
        max_depth=8,
        min_child_weight=20,
        subsample=0.80,
        colsample_bytree=0.80,
        reg_lambda=1.0,
        max_bin=256,
        tree_method="hist",
        scale_pos_weight=razao_desbalanceamento(y_treino_completo),
        n_jobs=-1,
        random_state=SEED,
    )
    modelo_xgb.fit(X_xgb, y_treino_completo, verbose=False)
    salvar_dicionario_dummies(preprocessador_xgb, "XGB")
    tempo_xgb = time.time() - inicio

    bundle_xgb = {
        "tipo": "XGB",
        "preprocessador": preprocessador_xgb,
        "modelo": modelo_xgb,
        "preditores": PREDITORES,
    }
    joblib.dump(
        bundle_xgb,
        os.path.join(PASTA_MODELOS, "XGB.joblib"),
        compress=3,
    )

    del X_xgb
    gc.collect()

    avaliar_modelo_2024(
        "XGB",
        bundle_xgb,
        tempo_xgb,
        {
            "n_estimators": n_arvores_xgb,
            "learning_rate": 0.05,
            "max_depth": 8,
            "min_child_weight": 20,
            "subsample": 0.80,
            "colsample_bytree": 0.80,
            "codificacao": "one-hot",
        },
    )

    del bundle_xgb, modelo_xgb, preprocessador_xgb
    gc.collect()

# %% [markdown]
# ## 12. LightGBM
#
# O LightGBM recebe as variáveis categóricas como categorias nativas.

# %%
if deve_executar("LGBM"):
    metadados_lgb_dev = aprender_metadados_nativos(X_dev)
    X_lgb_dev = preparar_lightgbm(X_dev, metadados_lgb_dev)
    X_lgb_2023 = preparar_lightgbm(X_2023, metadados_lgb_dev)

    modelo_lgb_dev = LGBMClassifier(
        objective="binary",
        n_estimators=2000,
        learning_rate=0.03,
        num_leaves=63,
        max_depth=-1,
        min_child_samples=100,
        subsample=0.80,
        subsample_freq=1,
        colsample_bytree=0.80,
        reg_lambda=1.0,
        scale_pos_weight=razao_desbalanceamento(y_dev),
        random_state=SEED,
        n_jobs=-1,
        verbosity=-1,
    )
    modelo_lgb_dev.fit(
        X_lgb_dev,
        y_dev,
        eval_set=[(X_lgb_2023, y_2023)],
        eval_metric="average_precision",
        categorical_feature=PREDITORES_CATEGORICOS,
        callbacks=[
            lgb.early_stopping(stopping_rounds=60, verbose=True),
            lgb.log_evaluation(period=100),
        ],
    )

    melhor_iteracao_lgb = getattr(modelo_lgb_dev, "best_iteration_", None)
    n_arvores_lgb = int(melhor_iteracao_lgb or 2000)
    print("Número de árvores escolhido para LGBM:", n_arvores_lgb)

    del (
        metadados_lgb_dev,
        X_lgb_dev,
        X_lgb_2023,
        modelo_lgb_dev,
    )
    gc.collect()

    inicio = time.time()
    metadados_lgb = aprender_metadados_nativos(X_treino_completo)
    X_lgb = preparar_lightgbm(X_treino_completo, metadados_lgb)

    modelo_lgb = LGBMClassifier(
        objective="binary",
        n_estimators=n_arvores_lgb,
        learning_rate=0.03,
        num_leaves=63,
        max_depth=-1,
        min_child_samples=100,
        subsample=0.80,
        subsample_freq=1,
        colsample_bytree=0.80,
        reg_lambda=1.0,
        scale_pos_weight=razao_desbalanceamento(y_treino_completo),
        random_state=SEED,
        n_jobs=-1,
        verbosity=-1,
    )
    modelo_lgb.fit(
        X_lgb,
        y_treino_completo,
        categorical_feature=PREDITORES_CATEGORICOS,
    )
    tempo_lgb = time.time() - inicio

    bundle_lgb = {
        "tipo": "LGBM",
        "metadados": metadados_lgb,
        "modelo": modelo_lgb,
        "preditores": PREDITORES,
    }
    joblib.dump(
        bundle_lgb,
        os.path.join(PASTA_MODELOS, "LGBM.joblib"),
        compress=3,
    )

    del X_lgb
    gc.collect()

    avaliar_modelo_2024(
        "LGBM",
        bundle_lgb,
        tempo_lgb,
        {
            "n_estimators": n_arvores_lgb,
            "learning_rate": 0.03,
            "num_leaves": 63,
            "min_child_samples": 100,
            "subsample": 0.80,
            "colsample_bytree": 0.80,
            "codificacao": "categorias nativas",
        },
    )

    del bundle_lgb, modelo_lgb, metadados_lgb
    gc.collect()

# %% [markdown]
# ## 13. CatBoost
#
# O CatBoost recebe diretamente os nomes das variáveis categóricas.

# %%
if deve_executar("CAT"):
    metadados_cat_dev = aprender_metadados_nativos(X_dev)
    X_cat_dev = preparar_catboost(X_dev, metadados_cat_dev)
    X_cat_2023 = preparar_catboost(X_2023, metadados_cat_dev)

    modelo_cat_dev = CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="PRAUC",
        iterations=1500,
        learning_rate=0.05,
        depth=8,
        l2_leaf_reg=3.0,
        random_seed=SEED,
        auto_class_weights="Balanced",
        thread_count=-1,
        allow_writing_files=False,
        verbose=100,
    )
    modelo_cat_dev.fit(
        X_cat_dev,
        y_dev,
        cat_features=PREDITORES_CATEGORICOS,
        eval_set=(X_cat_2023, y_2023),
        early_stopping_rounds=60,
        use_best_model=True,
    )

    melhor_iteracao_cat = modelo_cat_dev.get_best_iteration()
    n_arvores_cat = (
        int(melhor_iteracao_cat) + 1
        if melhor_iteracao_cat is not None and melhor_iteracao_cat >= 0
        else 1500
    )
    print("Número de árvores escolhido para CAT:", n_arvores_cat)

    del (
        metadados_cat_dev,
        X_cat_dev,
        X_cat_2023,
        modelo_cat_dev,
    )
    gc.collect()

    inicio = time.time()
    metadados_cat = aprender_metadados_nativos(X_treino_completo)
    X_cat = preparar_catboost(X_treino_completo, metadados_cat)

    modelo_cat = CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="PRAUC",
        iterations=n_arvores_cat,
        learning_rate=0.05,
        depth=8,
        l2_leaf_reg=3.0,
        random_seed=SEED,
        auto_class_weights="Balanced",
        thread_count=-1,
        allow_writing_files=False,
        verbose=100,
    )
    modelo_cat.fit(
        X_cat,
        y_treino_completo,
        cat_features=PREDITORES_CATEGORICOS,
    )
    tempo_cat = time.time() - inicio

    bundle_cat = {
        "tipo": "CAT",
        "metadados": metadados_cat,
        "modelo": modelo_cat,
        "preditores": PREDITORES,
    }
    joblib.dump(
        bundle_cat,
        os.path.join(PASTA_MODELOS, "CAT.joblib"),
        compress=3,
    )

    del X_cat
    gc.collect()

    avaliar_modelo_2024(
        "CAT",
        bundle_cat,
        tempo_cat,
        {
            "iterations": n_arvores_cat,
            "learning_rate": 0.05,
            "depth": 8,
            "l2_leaf_reg": 3.0,
            "auto_class_weights": "Balanced",
            "codificacao": "categorias nativas",
        },
    )

    del bundle_cat, modelo_cat, metadados_cat
    gc.collect()

# %% [markdown]
# ## 14. Tabela comparativa e escolha provisória
#
# A ordenação principal é feita por AP. O F1 abaixo é o melhor F1 obtido na
# validação de 2024 e não deve ser tratado como desempenho do teste final.

# %%
if not os.path.exists(ARQUIVO_METRICAS):
    raise FileNotFoundError("Nenhuma métrica foi produzida.")

metricas = pd.read_csv(ARQUIVO_METRICAS)

comparacao = (
    metricas.loc[
        metricas["Configuracao"] == "Threshold_F1_validacao_2024",
        [
            "Modelo",
            "N_validacao",
            "Prevalencia_2024",
            "Average_Precision_PR",
            "ROC_AUC",
            "Threshold",
            "Precision",
            "Recall_sensibilidade",
            "Especificidade",
            "F1",
            "Balanced_accuracy",
            "TN",
            "FP",
            "FN",
            "TP",
        ],
    ]
    .sort_values(
        ["Average_Precision_PR", "ROC_AUC", "F1"],
        ascending=False,
    )
    .reset_index(drop=True)
)

comparacao.insert(0, "Posicao_AP", np.arange(1, len(comparacao) + 1))
comparacao["AP_dividida_prevalencia"] = (
    comparacao["Average_Precision_PR"] / comparacao["Prevalencia_2024"]
)

ARQUIVO_COMPARACAO = os.path.join(
    PASTA_RESULTADOS, "02_comparacao_modelos_2024.csv"
)
comparacao.to_csv(ARQUIVO_COMPARACAO, index=False, encoding="utf-8-sig")

display(comparacao)

melhor_modelo = comparacao.iloc[0]["Modelo"]
print(f"\nMelhor modelo provisório por AP em 2024: {melhor_modelo}")
print("O teste de 2025 continua lacrado.")

# %% [markdown]
# ## 15. Gráfico comparativo

# %%
if len(comparacao):
    grafico = comparacao.set_index("Modelo")[
        ["Average_Precision_PR", "ROC_AUC", "F1"]
    ]
    ax = grafico.plot.bar(figsize=(10, 6), rot=0)
    ax.axhline(
        comparacao["Prevalencia_2024"].iloc[0],
        color="gray",
        linestyle="--",
        linewidth=1.5,
        label="Prevalência de 2024 / baseline da AP",
    )
    ax.set_ylim(0, 1)
    ax.set_ylabel("Valor")
    ax.set_title("Comparação dos modelos na validação temporal de 2024")
    ax.legend(loc="best")
    plt.tight_layout()
    plt.savefig(
        os.path.join(PASTA_RESULTADOS, "03_comparacao_metricas_2024.png"),
        dpi=200,
        bbox_inches="tight",
    )
    plt.show()

# %% [markdown]
# ## 16. Informações finais do experimento

# %%
informacoes_finais = {
    "experimento": "Comparação Dummy, LOGIT, RF, XGB, LGBM e CAT",
    "seed": SEED,
    "modo_dados": MODO_DADOS,
    "arquivo_amostra": ARQUIVO_AMOSTRA,
    "arquivos_treino_por_ano": ARQUIVOS_TREINO_POR_ANO,
    "n_treino_final": int(len(y_treino_completo)),
    "prevalencia_treino": float(np.mean(y_treino_completo)),
    "anos_desenvolvimento": ANOS_DESENVOLVIMENTO,
    "ano_early_stopping": ANO_EARLY_STOPPING,
    "ano_validacao": ANO_VALIDACAO,
    "ano_teste_final": 2025,
    "teste_2025_lido": False,
    "criterio_ordenacao": "Average Precision em 2024",
    "modelos_concluidos": comparacao["Modelo"].tolist(),
    "melhor_modelo_provisorio": melhor_modelo,
    "preditores_numericos": PREDITORES_NUMERICOS,
    "preditores_categoricos": PREDITORES_CATEGORICOS,
    "escolaridade_incluida": False,
    "vinculos_publicos_unidos": False,
    "codificacao_one_hot": ["LOGIT", "RF", "XGB"],
    "categorias_nativas": ["LGBM", "CAT"],
    "rf_max_samples": RF_MAX_SAMPLES,
}

with open(
    os.path.join(PASTA_RESULTADOS, "04_informacoes_experimento.json"),
    "w",
    encoding="utf-8",
) as arquivo_json:
    json.dump(informacoes_finais, arquivo_json, ensure_ascii=False, indent=2)

print("Arquivos produzidos em:")
print(PASTA_RESULTADOS)
print("\nExecução concluída. 2025 não foi utilizado.")


In [ ]:
import os
import time
import signal

try:
    from google.colab import runtime
    print("Solicitando desconexão graciosa da sessão...")
    runtime.unassign()

    # Aguarda 5 segundos para o backend do Colab processar e destruir a VM
    time.sleep(5)

    # Se o interpretador ainda estiver rodando após os 5 segundos, aciona o plano B
    print("Desconexão suave não respondeu. Forçando o encerramento do processo...")
    os.kill(os.getpid(), signal.SIGKILL)

except Exception as e:
    # Se a própria importação ou função do Colab gerar um erro, vai direto para o plano B
    print(f"Erro na biblioteca do Colab ({e}). Forçando a parada imediata...")
    os.kill(os.getpid(), signal.SIGKILL)